# Sea Surface Temperature and Salinity Bias
SST bias vs NOAA OISSTv2 and SSS bias vs JRA restore climatology.

Authors: Xinru Li, Brandon Reichl, and John Krasting

In [ ]:
# Development mode: constantly refreshes module code
%load_ext autoreload
%autoreload 2

## Framework Code and Diagnostic Setup

In [ ]:
import esnb
from esnb import NotebookDiagnostic, RequestedVariable, CaseGroup2, nbtools
from esnb.sites.gfdl import call_dmget, convert_to_momgrid
import climplot

In [ ]:
# Define requested variables: SST and SSS from ocean monthly output
variables = [
    RequestedVariable("tos", "ocean_month"),
    RequestedVariable("sos", "ocean_month"),
]

# Diagnostic configuration
diag_name = "SST and SSS Bias"
diag_desc = "Sea surface temperature and salinity bias analysis"
user_options = {"plot_region": ["global"]}

# Initialize the diagnostic
diag = NotebookDiagnostic(diag_name, diag_desc, variables=variables, **user_options)

In [ ]:
# Define experiments to analyze
groups = [
    CaseGroup2("odiv-1",     date_range=("0101-01-01", "0200-12-31"), name="CM4.0"),
    CaseGroup2("cmip6-1183", date_range=("0101-01-01", "0200-12-31"), name="ESM4.1"),
    CaseGroup2("esm45-122",  date_range=("0101-01-01", "0200-12-31"), name="ESM4.5 piControl"),
    CaseGroup2("esm45-148",  date_range=("0001-01-01", "0100-12-31"), name="ESM4.5 piControl branched"),
]

In [ ]:
# Combine experiments with the diagnostic request and determine files to load
diag.resolve(groups)

In [ ]:
# Print a list of file paths
_ = [print(x) for x in diag.files]

*(The files above are necessary to run the diagnostic.)*

In [ ]:
# Stage files from mass storage
call_dmget(diag.files, status=True)
call_dmget(diag.files)

In [ ]:
# Load the data as xarray datasets
diag.open()

In [ ]:
convert_to_momgrid(diag, return_corners=True)

## Main Diagnostic

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import xesmf as xe
import cartopy.crs as ccrs
import momlevel as ml
import momgrid

all_figs = []
climplot.publication()

In [ ]:
# ESM4 target grid for runtime SSS regridding
_esm4_grid = momgrid.MOMgrid("esm4_sym").to_xarray()

# Cache for ESM4 regridder to avoid recreating it
_esm4_regridder_cache = {}


def get_obs_key(model_type):
    """Map model type string to observation grid key."""
    if "esm4" in model_type:
        return "esm4"
    if "om4" in model_type:
        return "om4"
    return "om5"


# Load observational datasets once and cache them
_obs_cache = {}


def load_obs_datasets():
    """Load all observational datasets once and cache them."""
    global _obs_cache
    if _obs_cache:
        return _obs_cache
    
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    
    # Load SST data once per grid
    sst_data = {
        "om4": xr.open_dataset(
            "/archive/jpk/datasets/observations/NOAA-OISST/processed/"
            "NOAA_OISST_v2_anncycle_199101_202012_v20250722_OM4.nc",
            decode_times=time_coder,
        ),
        "om5": xr.open_dataset(
            "/archive/jpk/datasets/observations/NOAA-OISST/processed/"
            "NOAA_OISST_v2_anncycle_199101_202012_v20250722_OM5.nc",
            decode_times=time_coder,
        ),
        "esm4": xr.open_dataset(
            "/archive/jpk/datasets/observations/NOAA-OISST/processed/"
            "NOAA_OISST_v2_anncycle_199101_202012_v20250722_ESM4.nc",
            decode_times=time_coder,
        ),
    }
    
    # Load annual SST data once
    sst_annual_data = {
        "om4": xr.open_dataset(
            "/archive/jpk/datasets/observations/NOAA-OISST/processed/"
            "NOAA_OISST_v2_annual_199101_202012_v20250722_OM4.nc",
            decode_times=time_coder,
        ),
        "om5": xr.open_dataset(
            "/archive/jpk/datasets/observations/NOAA-OISST/processed/"
            "NOAA_OISST_v2_annual_199101_202012_v20250722_OM5.nc",
            decode_times=time_coder,
        ),
        "esm4": xr.open_dataset(
            "/archive/jpk/datasets/observations/NOAA-OISST/processed/"
            "NOAA_OISST_v2_annual_199101_202012_v20250722_ESM4.nc",
            decode_times=time_coder,
        ),
    }
    
    # Load JRA SSS data once per grid
    sss_jra_om4 = xr.open_dataset(
        "/archive/gold/datasets/OM4_025/INPUT.JRA.v2019.07.04/"
        "salt_restore_JRA.1440x1080.v20190706.nc", 
        decode_times=False,
    )
    sss_jra_om5 = xr.open_dataset(
        "/archive/gold/datasets/OM5_025/INPUT/salt_restore_JRA_v20240410.nc",
        decode_times=False,
    )
    
    _obs_cache = {
        "sst_data": sst_data,
        "sst_annual_data": sst_annual_data,
        "sss_jra_om4": sss_jra_om4,
        "sss_jra_om5": sss_jra_om5,
    }
    return _obs_cache


def _get_or_create_esm4_regridder(src_lon, src_lat):
    """Get cached ESM4 regridder or create a new one."""
    global _esm4_regridder_cache
    
    # Create a unique key from source grid shape
    cache_key = (src_lon.shape[0], src_lon.shape[1])
    
    if cache_key not in _esm4_regridder_cache:
        src = xr.Dataset({
            "lat": (["y", "x"], src_lat),
            "lon": (["y", "x"], src_lon),
        })
        tgt = xr.Dataset({
            "lat": (["y", "x"], _esm4_grid.geolat.values),
            "lon": (["y", "x"], _esm4_grid.geolon.values),
        })
        _esm4_regridder_cache[cache_key] = xe.Regridder(src, tgt, "bilinear", periodic=True)
    
    return _esm4_regridder_cache[cache_key]


def _regrid_jra_to_esm4(data, src_lon, src_lat):
    """Regrid a 2D or 3D JRA field from OM4 grid to ESM4 grid using cached regridder."""
    regridder = _get_or_create_esm4_regridder(src_lon, src_lat)
    
    _lon_attrs = {"standard_name": "longitude", "units": "degrees_east"}
    _lat_attrs = {"standard_name": "latitude", "units": "degrees_north"}
    
    if data.ndim == 2:
        da = xr.DataArray(data, dims=["y", "x"])
        out = regridder(da)
        result = xr.DataArray(
            out.values, dims=["yh", "xh"],
            coords={
                "yh": _esm4_grid.yh.values,
                "xh": _esm4_grid.xh.values,
                "geolon": (["yh", "xh"], _esm4_grid.geolon.values),
                "geolat": (["yh", "xh"], _esm4_grid.geolat.values),
            },
        )
        result.geolon.attrs.update(_lon_attrs)
        result.geolat.attrs.update(_lat_attrs)
        return result
    # 3D: (time, y, x)
    da = xr.DataArray(data, dims=["time", "y", "x"])
    out = regridder(da)
    result = xr.DataArray(
        out.values, dims=["time", "yh", "xh"],
        coords={
            "yh": _esm4_grid.yh.values,
            "xh": _esm4_grid.xh.values,
            "geolon": (["yh", "xh"], _esm4_grid.geolon.values),
            "geolat": (["yh", "xh"], _esm4_grid.geolat.values),
        },
    )
    result.geolon.attrs.update(_lon_attrs)
    result.geolat.attrs.update(_lat_attrs)
    return result


# Load all observational data once
obs_cache = load_obs_datasets()

# SST: NOAA OISSTv2 climatology (derived from cached annual cycle data)
SST_OBS_CLIMO = {
    "om4": momgrid.Gridset(obs_cache["sst_data"]["om4"]).data.tos.mean("time"),
    "om5": momgrid.Gridset(obs_cache["sst_data"]["om5"]).data.tos.mean("time"),
    "esm4": momgrid.Gridset(obs_cache["sst_data"]["esm4"]).data.tos.mean("time"),
}

# SST annual cycle obs (from cached data)
SST_OBS_ANNCYC = {
    "om4": momgrid.Gridset(obs_cache["sst_data"]["om4"]).data.tos,
    "om5": momgrid.Gridset(obs_cache["sst_data"]["om5"]).data.tos,
    "esm4": momgrid.Gridset(obs_cache["sst_data"]["esm4"]).data.tos,
}

# SSS: JRA salt restore climatology and annual cycle
# Process OM4 JRA data once
_om4_jra_sos = obs_cache["sss_jra_om4"].sos.rename({"i": "xh", "j": "yh"})
_om5_jra_sos = obs_cache["sss_jra_om5"].sos.rename({"i": "xh", "j": "yh"})

# Create ESM4 regridded version once (for both climo and anncyc)
_esm4_jra_sos = _regrid_jra_to_esm4(
    _om4_jra_sos.values, 
    _om4_jra_sos.lon.values, 
    _om4_jra_sos.lat.values,
)
_esm4_jra_sos = _esm4_jra_sos.assign_coords({"time": obs_cache["sss_jra_om4"].time})

SSS_OBS_CLIMO = {
    "om5": _om5_jra_sos.mean(dim="time"),
    "om4": _om4_jra_sos.mean(dim="time"),
    "esm4": _esm4_jra_sos.mean(dim="time"),
}

SSS_OBS_ANNCYC = {
    "om5": _om5_jra_sos,
    "om4": _om4_jra_sos,
    "esm4": _esm4_jra_sos,
}

In [ ]:
# Compute annual-mean model data for each group (lazy evaluation)
model_data = {}
for group in diag.groups:
    ds_tos = group.datasets[diag.varmap["tos"]]
    ds_sos = group.datasets[diag.varmap["sos"]]
    model_type = ds_tos.attrs.get("model", "")
    obs_key = get_obs_key(model_type)

    model_data[group] = {
        "tos": ml.util.annual_average(ds_tos.tos).mean("time", keep_attrs=True),
        "sos": ml.util.annual_average(ds_sos.sos).mean("time", keep_attrs=True),
        "obs_key": obs_key,
        "ds_tos": ds_tos,
        "ds_sos": ds_sos,
    }

In [ ]:
REGIONS = {
    "global": {
        "projection": ccrs.Robinson(central_longitude=-160),
        "sst_levels": (-4.0, 4.25, 0.25),
        "sss_levels": (-2.0, 2.1, 0.1),
        "xrange": None,
        "yrange": None,
    },
    "arctic": {
        "projection": ccrs.NorthPolarStereo(),
        "sst_levels": (-2.0, 2.125, 0.125),
        "sss_levels": (-2.0, 2.125, 0.125),
        "xrange": (-298, 61),
        "yrange": (60.0, 91.0),
    },
    "southern_ocean": {
        "projection": ccrs.SouthPolarStereo(),
        "sst_levels": (-2.0, 2.125, 0.125),
        "sss_levels": (-2.0, 2.125, 0.125),
        "xrange": (-300, 60),
        "yrange": (-60.0, -91.0),
    },
    "nw_pacific": {
        "projection": ccrs.Miller(central_longitude=-200),
        "sst_levels": (-4.5, 4.75, 0.25),
        "sss_levels": (-2.0, 2.1, 0.1),
        "xrange": (-250, -150),
        "yrange": (25, 60),
    },
    "trop_indopac": {
        "projection": ccrs.Miller(central_longitude=-180),
        "sst_levels": (-4.5, 4.75, 0.25),
        "sss_levels": (-2.0, 2.1, 0.1),
        "xrange": (-280, -80),
        "yrange": (-23, 23),
    },
    "australia": {
        "projection": ccrs.Miller(central_longitude=-180),
        "sst_levels": (-4.5, 4.75, 0.25),
        "sss_levels": (-2.0, 2.1, 0.1),
        "xrange": (-300, -170),
        "yrange": (0, -60),
    },
    "north_atlantic": {
        "projection": ccrs.Miller(central_longitude=-60),
        "sst_levels": (-4.5, 4.75, 0.25),
        "sss_levels": (-2.0, 2.1, 0.1),
        "xrange": (-80, 0),
        "yrange": (30, 75),
    },
    "south_atlantic": {
        "projection": ccrs.Miller(central_longitude=-25),
        "sst_levels": (-4.5, 4.75, 0.25),
        "sss_levels": (-2.0, 2.1, 0.1),
        "xrange": (-75, 25),
        "yrange": (-20, -60),
    },
    "california": {
        "projection": ccrs.Miller(central_longitude=-115),
        "sst_levels": (-4.5, 4.75, 0.25),
        "sss_levels": (-2.0, 2.1, 0.1),
        "xrange": (-130, -100),
        "yrange": (18, 46),
    },
    "bengula": {
        "projection": ccrs.Miller(central_longitude=5),
        "sst_levels": (-4.5, 4.75, 0.25),
        "sss_levels": (-2.0, 2.1, 0.1),
        "xrange": (-10, 20),
        "yrange": (35, -30),
    },
    "peru": {
        "projection": ccrs.Miller(central_longitude=-85),
        "sst_levels": (-4.5, 4.75, 0.25),
        "sss_levels": (-2.0, 2.1, 0.1),
        "xrange": (-110, -60),
        "yrange": (0, -40),
    },
    "caribbean": {
        "projection": ccrs.Miller(central_longitude=-80),
        "sst_levels": (-4.5, 4.75, 0.25),
        "sss_levels": (-2.0, 2.1, 0.1),
        "xrange": (-110, -60),
        "yrange": (5, 35),
    },
}

In [ ]:
for reg_name, reg_cfg in REGIONS.items():
    do_plot = reg_name in diag.diag_vars["plot_region"]
    nexps = len(diag.groups)

    all_stats = {}

    # Create two separate figures: one for SST and one for SSS
    for varname, obs_dict, levels_key, unit_label in [
        ("tos", SST_OBS_CLIMO, "sst_levels", "SST Bias [degC]"),
        ("sos", SSS_OBS_CLIMO, "sss_levels", "SSS Bias [PSU]"),
    ]:
        if do_plot:
            # Use nbtools to get appropriate figure size and subplot layout
            figsize, subplot = nbtools.get_figsize_subplots(nexps)
            fig, axes = plt.subplots(
                *subplot, figsize=figsize, dpi=150,
                subplot_kw={"projection": reg_cfg["projection"]},
            )
            axes = np.atleast_1d(axes).flatten()

        cmap, norm, _ = climplot.discrete_cmap(
            *reg_cfg[levels_key], cmap_name="RdBu_r", center_on_white=True,
        )

        for n, group in enumerate(diag.groups):
            d = model_data[group]
            model_arr = d[varname]
            obs_arr = obs_dict[d["obs_key"]]
            ds = d[f"ds_{varname}"]

            # Subset if regional
            if reg_cfg["xrange"] is not None:
                model_arr = momgrid.geoslice.geoslice(
                    model_arr, x=reg_cfg["xrange"], y=reg_cfg["yrange"],
                )
                obs_arr = momgrid.geoslice.geoslice(
                    obs_arr, x=reg_cfg["xrange"], y=reg_cfg["yrange"],
                )

            # Compute stats
            stats, stats_str = nbtools.calculate_stats(
                model_arr, obs_arr, model_arr.areacello,
            )
            for k, v in stats.items():
                group.add_metric(f"{varname}_{reg_name}", (k, float(v)))

            # Plot
            if do_plot:
                ax = axes[n]
                ax.set_facecolor("#808080")
                if reg_cfg["xrange"] is None:
                    cb = climplot.plot_ocean_field(
                        ax, ds.geolon_c.values, ds.geolat_c.values,
                        (model_arr - obs_arr).values, cmap=cmap, norm=norm,
                    )
                else:
                    cb = ax.pcolormesh(
                        model_arr.geolon, model_arr.geolat,
                        model_arr - obs_arr,
                        transform=ccrs.PlateCarree(), cmap=cmap, norm=norm,
                    )
                    ax.coastlines(linewidth=0.5)
                title_var = "SST" if varname == "tos" else "SSS"
                ax.set_title(f"{group.name}", fontsize=8)
                # Fixed: Changed fontsize from 5 to 7, position from y=-0.1 to y=1.03
                ax.text(
                    0.99, 1.03, stats_str, ha="right", style="italic",
                    transform=ax.transAxes, fontsize=7,
                )

        if do_plot:
            # Hide any unused subplots
            for idx in range(nexps, len(axes)):
                axes[idx].axis('off')
            
            # Add colorbar at bottom
            cbar = nbtools.bottom_colorbar(fig, cb, orientation="horizontal", extend="both")
            cbar.set_label(unit_label, fontsize=7)
            
            title_var = "SST" if varname == "tos" else "SSS"
            fig.suptitle(f"{reg_name.replace('_', ' ').title()} - {title_var} Bias", y=1.01)
            all_figs.append(fig)

### Seasonal Cycle

In [ ]:
# Compute model annual cycles (lazy evaluation)
anncyc_data = {}
for group in diag.groups:
    ds_tos = group.datasets[diag.varmap["tos"]]
    ds_sos = group.datasets[diag.varmap["sos"]]
    model_type = ds_tos.attrs.get("model", "")
    obs_key = get_obs_key(model_type)
    anncyc_data[group] = {
        "tos": ml.util.annual_cycle(ds_tos.tos),
        "sos": ml.util.annual_cycle(ds_sos.sos),
        "obs_key": obs_key,
        "ds_tos": ds_tos,
        "ds_sos": ds_sos,
    }

In [ ]:
# Zonal-mean lat-month Hovmoller of SST and SSS bias - separate figures
nexps = len(diag.groups)
time_months = np.arange(1, 13)

# Create two separate figures: one for SST and one for SSS
for varname, obs_anncyc, unit_label in [
    ("tos", SST_OBS_ANNCYC, "SST Bias [degC]"),
    ("sos", SSS_OBS_ANNCYC, "SSS Bias [PSU]"),
]:
    # Use nbtools to get appropriate figure size and subplot layout
    figsize, subplot = nbtools.get_figsize_subplots(nexps)
    fig, axes = plt.subplots(*subplot, figsize=figsize, dpi=150)
    axes = np.atleast_1d(axes).flatten()
    
    for n, group in enumerate(diag.groups):
        d = anncyc_data[group]
        model_clim = d[varname]
        obs_clim = obs_anncyc[d["obs_key"]]
        ds = d[f"ds_{varname}"]
        area = ds.areacello
        lat = ds.geolat.mean("xh")

        model_zm = model_clim.assign_coords({"time": time_months})
        model_zm = model_zm.weighted(area).mean("xh")
        obs_zm = obs_clim.assign_coords({"time": time_months})
        obs_zm = obs_zm.weighted(area).mean("xh")
        diff = model_zm - obs_zm

        # Calculate statistics for zonal-mean seasonal cycle
        stats, stats_str = nbtools.calculate_stats(
            model_zm, obs_zm, area.mean("xh"),  # Use zonal-mean area weights
        )
        # Register metrics
        for k, v in stats.items():
            group.add_metric(f"{varname}_seasonal_cycle", (k, float(v)))

        ax = axes[n]
        ax.set_facecolor("gray")
        levels = np.arange(-1, 1.1, 0.1)
        cb = ax.contourf(
            diff.time, lat, diff.T, levels=levels, cmap="RdBu_r", extend="both",
        )
        title_var = "SST" if varname == "tos" else "SSS"
        ax.set_title(f"{group.name}", fontsize=8)
        ax.set_xticks(np.arange(1, 13))
        ax.set_xticklabels(
            ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"],
        )
        ax.set_ylim(-78, None)
        ax.grid(True, color="k", linestyle=":", linewidth=0.5)
        ax.set_ylabel("Latitude", fontsize=7)
        ax.set_xlabel("Month", fontsize=7)
        # Display metrics on plot
        ax.text(
            0.99, 1.03, stats_str, ha="right", style="italic",
            transform=ax.transAxes, fontsize=7,
        )

    # Hide any unused subplots
    for idx in range(nexps, len(axes)):
        axes[idx].axis('off')
    
    # Add colorbar at bottom
    cbar = nbtools.bottom_colorbar(fig, cb, orientation="horizontal", extend="both")
    cbar.set_label(unit_label, fontsize=7)
    
    title_var = "SST" if varname == "tos" else "SSS"
    fig.suptitle(f"Zonal Mean {title_var} Bias by Month", y=1.01)
    all_figs.append(fig)

### Maps by Season

In [ ]:
seasons = {
    "DJF": [11, 0, 1],
    "MAM": [2, 3, 4],
    "JJA": [5, 6, 7],
    "SON": [8, 9, 10],
}

for season, months in seasons.items():
    nexps = len(diag.groups)
    
    # Create two separate figures: one for SST and one for SSS
    for varname, obs_anncyc, levels, unit_label in [
        ("tos", SST_OBS_ANNCYC, (-7, 7.5, 0.5), "SST Bias [degC]"),
        ("sos", SSS_OBS_ANNCYC, (-3, 3.25, 0.25), "SSS Bias [PSU]"),
    ]:
        # Use nbtools to get appropriate figure size and subplot layout
        figsize, subplot = nbtools.get_figsize_subplots(nexps)
        fig, axes = plt.subplots(
            *subplot, figsize=figsize, dpi=150,
            subplot_kw={"projection": ccrs.Robinson(central_longitude=-160)},
        )
        axes = np.atleast_1d(axes).flatten()

        cmap, norm, _ = climplot.discrete_cmap(
            *levels, cmap_name="RdBu_r", center_on_white=True,
        )

        for n, group in enumerate(diag.groups):
            d = anncyc_data[group]
            model_seas = d[varname].isel(time=months).mean("time")
            obs_seas = obs_anncyc[d["obs_key"]].isel(time=months).mean("time")
            ds = d[f"ds_{varname}"]

            ax = axes[n]
            cb = climplot.plot_ocean_field(
                ax, ds.geolon_c.values, ds.geolat_c.values,
                (model_seas - obs_seas).values, cmap=cmap, norm=norm,
            )
            ax.set_title(f"{group.name}", fontsize=8)

            # Register seasonal stats and capture the string for display
            stats, stats_str = nbtools.calculate_stats(
                model_seas, obs_seas, model_seas.areacello,
            )
            for k, v in stats.items():
                group.add_metric(
                    f"{varname}_{season.lower()}_bias", (k, float(v)),
                )
            # Display metrics on plot
            ax.text(
                0.99, 1.03, stats_str, ha="right", style="italic",
                transform=ax.transAxes, fontsize=7,
            )

        # Hide any unused subplots
        for idx in range(nexps, len(axes)):
            axes[idx].axis('off')
        
        # Add colorbar at bottom
        cbar = nbtools.bottom_colorbar(fig, cb, orientation="horizontal", extend="both")
        cbar.set_label(unit_label, fontsize=7)
        
        title_var = "SST" if varname == "tos" else "SSS"
        fig.suptitle(f"{season} - {title_var} Bias", y=1.01)
        all_figs.append(fig)

### SST and SSS Trends

In [ ]:
# SST annual obs (from cached data)
_obs_annual_gs = {
    "om4": momgrid.Gridset(obs_cache["sst_annual_data"]["om4"]),
    "om5": momgrid.Gridset(obs_cache["sst_annual_data"]["om5"]),
    "esm4": momgrid.Gridset(obs_cache["sst_annual_data"]["esm4"]),
}
SST_OBS_ANNUAL = {k: v.data.tos for k, v in _obs_annual_gs.items()}

# Observed SST trend
obs_sst_slope = ml.trend.calc_linear_trend(
    SST_OBS_ANNUAL["om5"], time_units="yr",
)["tos_slope"]

# Model trends for both variables (no .load() to keep lazy)
model_trends = {}
for group in diag.groups:
    ds_tos = group.datasets[diag.varmap["tos"]]
    ds_sos = group.datasets[diag.varmap["sos"]]
    tos_ann = ml.util.annual_average(ds_tos.tos)
    sos_ann = ml.util.annual_average(ds_sos.sos)
    tos_trend = ml.trend.calc_linear_trend(tos_ann, time_units="yr")["tos_slope"]
    sos_trend = ml.trend.calc_linear_trend(sos_ann, time_units="yr")["sos_slope"]
    tos_trend = tos_trend.assign_coords({
        "geolon": ds_tos.geolon, "geolat": ds_tos.geolat,
        "areacello": ds_tos.areacello,
    })
    sos_trend = sos_trend.assign_coords({
        "geolon": ds_sos.geolon, "geolat": ds_sos.geolat,
        "areacello": ds_sos.areacello,
    })
    model_trends[group] = {"tos": tos_trend, "sos": sos_trend}

In [ ]:
# Plot observed SST trend
fig, ax = climplot.map_figure(central_longitude=-160)
cmap, norm, _ = climplot.discrete_cmap(
    -0.1, 0.11, 0.01, cmap_name="RdBu_r", center_on_white=True,
)
cb = climplot.plot_ocean_field(
    ax, _obs_annual_gs["om5"].grid.geolon_c,
    _obs_annual_gs["om5"].grid.geolat_c,
    obs_sst_slope.values, cmap=cmap, norm=norm,
)
ax.set_title("NOAA OISSTv2 SST Trend")
climplot.add_colorbar(cb, ax, "SST Trend [degC yr$^{-1}$]")
all_figs.append(fig)

In [ ]:
for reg_name, reg_cfg in REGIONS.items():
    do_plot = reg_name in diag.diag_vars["plot_region"]
    nexps = len(diag.groups)

    # Create two separate figures: one for SST trends and one for SSS trends
    for varname, trend_levels, unit_label in [
        ("tos", (-0.1, 0.11, 0.01), "SST Trend [degC yr$^{-1}$]"),
        ("sos", (-0.01, 0.011, 0.001), "SSS Trend [PSU yr$^{-1}$]"),
    ]:
        if do_plot:
            # Use nbtools to get appropriate figure size and subplot layout
            figsize, subplot = nbtools.get_figsize_subplots(nexps)
            fig, axes = plt.subplots(
                *subplot, figsize=figsize, dpi=150,
                subplot_kw={"projection": reg_cfg["projection"]},
            )
            axes = np.atleast_1d(axes).flatten()

        cmap, norm, _ = climplot.discrete_cmap(
            *trend_levels, cmap_name="RdBu_r", center_on_white=True,
        )

        for n, group in enumerate(diag.groups):
            arr = model_trends[group][varname]
            if reg_cfg["xrange"] is not None:
                arr = momgrid.geoslice.geoslice(
                    arr, x=reg_cfg["xrange"], y=reg_cfg["yrange"],
                )
            avg_trend = float(arr.weighted(arr.areacello).mean(("yh", "xh")))
            group.add_metric(f"{varname}_{reg_name}_trend", ("trend", avg_trend))

            if do_plot:
                ax = axes[n]
                ax.set_facecolor("#808080")
                if reg_cfg["xrange"] is None:
                    ds = model_data[group][f"ds_{varname}"]
                    cb = climplot.plot_ocean_field(
                        ax, ds.geolon_c.values, ds.geolat_c.values,
                        arr.values, cmap=cmap, norm=norm,
                    )
                else:
                    cb = ax.pcolormesh(
                        arr.geolon, arr.geolat, arr,
                        transform=ccrs.PlateCarree(), cmap=cmap, norm=norm,
                    )
                    ax.coastlines(linewidth=0.5)
                ax.set_title(f"{group.name}", fontsize=8)

        if do_plot:
            # Hide any unused subplots
            for idx in range(nexps, len(axes)):
                axes[idx].axis('off')
            
            # Add colorbar at bottom
            cbar = nbtools.bottom_colorbar(fig, cb, orientation="horizontal", extend="both")
            cbar.set_label(unit_label, fontsize=7)
            
            title_var = "SST" if varname == "tos" else "SSS"
            fig.suptitle(
                f"{reg_name.replace('_', ' ').title()} - {title_var} Trends", y=1.01,
            )
            all_figs.append(fig)

### Process and Synthesize Metrics

In [ ]:
import pandas as pd

data = diag.metrics
exps = list(data["RESULTS"]["Global"].keys())
metrics = list(data["RESULTS"]["Global"][exps[0]].keys())
trends = set([x for x in metrics if "_trend" in x])
seasonal = set([x for x in metrics if "_bias" in x])
regions = set(metrics) - trends - seasonal

# Regional bias summary
results = {}
for exp in exps:
    results[exp] = {}
    for reg in sorted(regions):
        for varname in ["tos", "sos"]:
            key = f"{varname}_{reg.replace(varname + '_', '')}" if varname in reg else reg
            if key in data["RESULTS"]["Global"][exp]:
                bias_val = data["RESULTS"]["Global"][exp][key].get("bias", None)
                if bias_val is not None:
                    results[exp][key] = bias_val

print("Regional Bias Summary:")
display(pd.DataFrame(results))

# Trend summary
results = {}
for exp in exps:
    results[exp] = {}
    for trend in sorted(trends):
        results[exp][trend] = data["RESULTS"]["Global"][exp][trend]["trend"]

print("\nTrend Summary:")
display(pd.DataFrame(results))

# Seasonal summary
results = {}
for exp in exps:
    results[exp] = {}
    for seas in sorted(seasonal):
        for varname in ["tos", "sos"]:
            if varname in seas:
                bias_val = data["RESULTS"]["Global"][exp][seas].get("bias", None)
                if bias_val is not None:
                    results[exp][seas] = bias_val

print("\nSeasonal Summary:")
display(pd.DataFrame(results))

### Write Metrics to File

In [ ]:
diag.write_metrics("SST_SSS_bias_metrics.json")

### Make a PowerPoint Presentation of Figures

In [ ]:
nbtools.save_pptx(all_figs, "SST_SSS_bias.pptx")